In [16]:
import os, sys
import importlib
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

from qtrans.utils import qutils
from qtrans.utils.url_utils import encode_quirk_url
qutils = importlib.reload(qutils)

parse_quirk_url = qutils.parse_quirk_url
quirk_to_qasm = qutils.quirk_to_qasm
quirk_circuit_info = qutils.quirk_circuit_info
append_custom_gates = qutils.append_custom_gates


In [17]:
import numpy as np
import urllib.parse
import json

def matriz_a_url_quirk(U):
    """
    Descompone una matriz 2x2 en compuertas Z^t, Y^t, Z^t y genera la URL de Quirk.
    """
    U = np.array(U, dtype=complex)
    
    # 1. Verificar si es unitaria
    if not np.allclose(np.eye(2), np.dot(U, U.conj().T)):
        print("⚠️ Advertencia: La matriz no es estrictamente unitaria.")

    # 2. Descomposición ZYZ
    det = np.linalg.det(U)
    gamma = np.angle(det) / 2.0
    U_su2 = U * np.exp(-1j * gamma)
    
    cos_half_theta = np.abs(U_su2[0, 0])
    sin_half_theta = np.abs(U_su2[1, 0])
    theta = 2 * np.arctan2(sin_half_theta, cos_half_theta)
    
    if np.isclose(sin_half_theta, 0):
        phi = 0.0
        lambda_ = 2 * np.angle(U_su2[1, 1])
    else:
        arg1 = np.angle(U_su2[1, 1])
        arg2 = np.angle(U_su2[1, 0])
        phi = arg1 + arg2
        lambda_ = arg1 - arg2

    # Normalizar ángulos entre -pi y pi
    angles = (np.array([theta, phi, lambda_]) + np.pi) % (2 * np.pi) - np.pi
    theta_norm, phi_norm, lambda_norm = angles

    # 3. Convertir ángulos a exponentes 't' de Quirk (t = ángulo / pi)
    # Redondeamos a 8 decimales para conservar precisión sin alargar demasiado la URL
    t_z1 = round(lambda_norm / np.pi, 8)
    t_y  = round(theta_norm / np.pi, 8)
    t_z2 = round(phi_norm / np.pi, 8)

    # 4. Usar IDs nativos cuando existan y matrices personalizadas para otros ángulos
    compuertas_personalizadas = []

    def numero_quirk(valor):
        real = float(np.real(valor))
        imaginario = float(np.imag(valor))
        if np.isclose(real, 0, atol=1e-12):
            real = 0.0
        if np.isclose(imaginario, 0, atol=1e-12):
            imaginario = 0.0
        if imaginario == 0:
            return f"{real:.10g}"
        if real == 0:
            return f"{imaginario:.10g}i"
        return f"{real:.10g}{imaginario:+.10g}i"

    def compuerta_quirk(eje, exponente):
        if np.isclose(exponente, 0):
            return 1
        if np.isclose(abs(exponente), 1):
            return eje
        fracciones = {0.5: '½', 0.25: '¼'}
        fraccion = next((simbolo for valor, simbolo in fracciones.items()
                         if np.isclose(abs(exponente), valor)), None)
        if fraccion is not None:
            signo = '-' if exponente < 0 else ''
            return f"{eje}^{signo}{fraccion}"

        angulo = np.pi * exponente
        fase = np.exp(1j * angulo / 2)
        coseno = np.cos(angulo / 2)
        seno = np.sin(angulo / 2)
        if eje == 'Y':
            matriz = [[fase * coseno, -fase * seno],
                      [fase * seno, fase * coseno]]
        elif eje == 'Z':
            matriz = [[1, 0], [0, np.exp(1j * angulo)]]
        else:
            raise ValueError(f"Eje de rotación no soportado: {eje}")

        filas = [','.join(numero_quirk(valor) for valor in fila) for fila in matriz]
        matriz_quirk = '{{' + '},{'.join(filas) + '}}'
        gate_id = f"~rot{len(compuertas_personalizadas)}"
        compuertas_personalizadas.append({
            "id": gate_id,
            "name": f"{eje}^{exponente:.8f}",
            "matrix": matriz_quirk
        })
        return gate_id

    compuerta_z1 = compuerta_quirk('Z', t_z1)
    compuerta_y  = compuerta_quirk('Y', t_y)
    compuerta_z2 = compuerta_quirk('Z', t_z2)

    # Crear las columnas del circuito (las operaciones consecutivas van en columnas separadas)
    # Recuerda que se aplican de derecha a izquierda matemáticamente, 
    # por lo que en el orden visual de Quirk van: z1 -> y -> z2
    columnas = []
    if compuerta_z1 != 1: columnas.append([compuerta_z1])
    if compuerta_y  != 1: columnas.append([compuerta_y])
    if compuerta_z2 != 1: columnas.append([compuerta_z2])
    
    # Si todas resultaron ser la identidad (0), dejamos una columna vacía
    if not columnas:
        columnas = [[1]]

    # 5. Estructurar el JSON de Quirk y codificarlo para la URL
    objeto_quirk = {"cols": columnas}
    if compuertas_personalizadas:
        objeto_quirk["gates"] = compuertas_personalizadas
    json_string = json.dumps(objeto_quirk, separators=(',', ':'))
    url_encoded = urllib.parse.quote(json_string, safe='')
    
    url_final = f"https://algassert.com/quirk#circuit={url_encoded}"
    return url_final



In [18]:
from urllib.parse import unquote
url=unquote("https://algassert.com/quirk#circuit={%22cols%22:[[%22~2onn%22]],%22gates%22:[{%22id%22:%22~2onn%22,%22name%22:%22x20%22,%22matrix%22:%22{{0.9848078,-0.1736482i},{-0.1736482i,0.9848078}}%2２}]}")

print(url)

https://algassert.com/quirk#circuit={"cols":[["~2onn"]],"gates":[{"id":"~2onn","name":"x20","matrix":"{{0.9848078,-0.1736482i},{-0.1736482i,0.9848078}}%2２}]}


In [19]:
import numpy as np

matrix = np.array([
    [0.9848078, -0.1736482j],
    [-0.1736482j, 0.9848078]
], dtype=complex)

In [20]:
from IPython.display import HTML, display
from urllib.parse import unquote

# --- EJEMPLO DE PRUEBA ---
# Probemos con una matriz de rotación custom aleatoria o una Hadamard:
url = matriz_a_url_quirk(matrix)


print("🔗 Copia y pega este enlace en tu navegador para ver tus compuertas en Quirk:")
print(url)


encoded_url = encode_quirk_url(url)

print(encoded_url)


print("Original en Quirk:", unquote(url))
display(HTML('<a href="' + encode_quirk_url(encoded_url) + '" target="_blank">Abrir en Quirk</a>'))


🔗 Copia y pega este enlace en tu navegador para ver tus compuertas en Quirk:
https://algassert.com/quirk#circuit=%7B%22cols%22%3A%5B%5B%22Z%5E%5Cu00bd%22%5D%2C%5B%22~rot0%22%5D%2C%5B%22Z%5E-%5Cu00bd%22%5D%5D%2C%22gates%22%3A%5B%7B%22id%22%3A%22~rot0%22%2C%22name%22%3A%22Y%5E0.11111112%22%2C%22matrix%22%3A%22%7B%7B0.9698463056%2B0.1710100848i%2C-0.1710100848-0.03015369438i%7D%2C%7B0.1710100848%2B0.03015369438i%2C0.9698463056%2B0.1710100848i%7D%7D%22%7D%5D%7D
https://algassert.com/quirk#circuit=%7B%22cols%22%3A%5B%5B%22Z%5E%5Cu00bd%22%5D%2C%5B%22~rot0%22%5D%2C%5B%22Z%5E-%5Cu00bd%22%5D%5D%2C%22gates%22%3A%5B%7B%22id%22%3A%22~rot0%22%2C%22name%22%3A%22Y%5E0.11111112%22%2C%22matrix%22%3A%22%7B%7B0.9698463056%2B0.1710100848i%2C-0.1710100848-0.03015369438i%7D%2C%7B0.1710100848%2B0.03015369438i%2C0.9698463056%2B0.1710100848i%7D%7D%22%7D%5D%7D
Original en Quirk: https://algassert.com/quirk#circuit={"cols":[["Z^\u00bd"],["~rot0"],["Z^-\u00bd"]],"gates":[{"id":"~rot0","name":"Y^0.11111112","matri

In [21]:
import numpy as np

def traducir_lista_a_quirk_gates(lista_matrices):
    """
    Recibe una lista de diccionarios con matrices de 2x2 y las descompone
    de forma numéricamente exacta en compuertas nativas de Quirk [Z^t, Y^t, Z^t].
    """
    resultados_traduccion = {}

    for item in lista_matrices:
        nombre = item["name"]
        U = np.array(item["matrix"], dtype=complex)
        
        # 1. Verificar unitariedad
        if not np.allclose(np.eye(2), np.dot(U, U.conj().T), atol=1e-5):
            print(f"⚠️ Advertencia: La matriz {nombre} no es estrictamente unitaria.")
            
        # 2. Forzar a SU(2) dividiendo por la raíz del determinante
        det = np.linalg.det(U)
        U_su2 = U / np.sqrt(det)
        
        # Extraer componentes canónicos de la primera fila
        a = U_su2[0, 0]
        b = U_su2[0, 1]
        
        # 3. Calcular Ángulo de Rotación Principal (Theta)
        theta = 2 * np.arctan2(np.abs(b), np.abs(a))
        
        # 4. Calcular Ángulos de Fase (Phi y Lambda) de forma robusta
        if np.isclose(np.abs(b), 0):
            # Caso especial: Rotación pura en Z (Evita indeterminación)
            phi = 0.0
            lambda_ = -2 * np.angle(a)
        else:
            # Extraer las fases directamente sin perder el signo algebraico
            angle_a = np.angle(a)
            angle_minus_b = np.angle(-b)
            
            phi = -angle_a - angle_minus_b
            lambda_ = -angle_a + angle_minus_b

        # 5. Normalizar ángulos en el rango [-pi, pi]
        angles = np.mod([theta, phi, lambda_], 2 * np.pi)
        for i in range(3):
            if angles[i] > np.pi:
                angles[i] -= 2 * np.pi
        
        theta_norm, phi_norm, lambda_norm = angles

        # 6. Convertir a exponentes fraccionarios t de Quirk (t = Ángulo / pi)
        t_z1 = round(lambda_norm / np.pi, 4)
        t_y  = round(theta_norm / np.pi, 4)
        t_z2 = round(phi_norm / np.pi, 4)

        # 7. Construir secuencia en orden del timeline de Quirk (Z1 -> Y -> Z2)
        secuencia_gates = []
        if not np.isclose(t_z1, 0, atol=1e-4): secuencia_gates.append(f"Z^{t_z1}")
        if not np.isclose(t_y, 0, atol=1e-4):  secuencia_gates.append(f"Y^{t_y}")
        if not np.isclose(t_z2, 0, atol=1e-4): secuencia_gates.append(f"Z^{t_z2}")
        
        resultados_traduccion[nombre] = secuencia_gates if secuencia_gates else ["Identidad"]

    return resultados_traduccion

In [22]:
import numpy as np

def traducir_lista_a_quirk_gates(lista_matrices):
    """
    Recibe una lista de diccionarios con matrices de 2x2 y las descompone
    de forma numéricamente exacta en compuertas nativas de Quirk [Z^t, Y^t, Z^t].
    """
    resultados_traduccion = {}

    for item in lista_matrices:
        nombre = item["name"]
        U = np.array(item["matrix"], dtype=complex)
        
        # 1. Verificar unitariedad
        if not np.allclose(np.eye(2), np.dot(U, U.conj().T), atol=1e-5):
            print(f"⚠️ Advertencia: La matriz {nombre} no es estrictamente unitaria.")
            
        # 2. Forzar a SU(2) dividiendo por la raíz del determinante
        det = np.linalg.det(U)
        U_su2 = U / np.sqrt(det)
        
        # Extraer componentes canónicos de la primera fila
        a = U_su2[0, 0]
        b = U_su2[0, 1]
        
        # 3. Calcular Ángulo de Rotación Principal (Theta)
        theta = 2 * np.arctan2(np.abs(b), np.abs(a))
        
        # 4. Calcular Ángulos de Fase (Phi y Lambda) de forma robusta
        if np.isclose(np.abs(b), 0):
            # Caso especial: Rotación pura en Z (Evita indeterminación)
            phi = 0.0
            lambda_ = -2 * np.angle(a)
        else:
            # Extraer las fases directamente sin perder el signo algebraico
            angle_a = np.angle(a)
            angle_minus_b = np.angle(-b)
            
            phi = -angle_a - angle_minus_b
            lambda_ = -angle_a + angle_minus_b

        # 5. Normalizar ángulos en el rango [-pi, pi]
        angles = np.mod([theta, phi, lambda_], 2 * np.pi)
        for i in range(3):
            if angles[i] > np.pi:
                angles[i] -= 2 * np.pi
        
        theta_norm, phi_norm, lambda_norm = angles

        # 6. Convertir a exponentes fraccionarios t de Quirk (t = Ángulo / pi)
        t_z1 = round(lambda_norm / np.pi, 4)
        t_y  = round(theta_norm / np.pi, 4)
        t_z2 = round(phi_norm / np.pi, 4)

        # 7. Construir secuencia en orden del timeline de Quirk (Z1 -> Y -> Z2)
        secuencia_gates = []
        if not np.isclose(t_z1, 0, atol=1e-4): secuencia_gates.append(f"Z^{t_z1}")
        if not np.isclose(t_y, 0, atol=1e-4):  secuencia_gates.append(f"Y^{t_y}")
        if not np.isclose(t_z2, 0, atol=1e-4): secuencia_gates.append(f"Z^{t_z2}")
        
        resultados_traduccion[nombre] = secuencia_gates if secuencia_gates else ["Identidad"]

    return resultados_traduccion


In [23]:
# --- PRUEBA CON TU DATASET COMPLETO ---
# # Definición de tus 7 matrices en NumPy
x20 = np.array([[0.9848078, -0.1736482j], [-0.1736482j, 0.9848078]])
xy20 = np.array([[0.9848078, -0.1227878 - 0.1227878j], [0.1227878 - 0.1227878j, 0.9848078]])
xyz20 = np.array([[0.9848078 - 0.1002558j, -0.1002558 - 0.1002558j], [0.1002558 - 0.1002558j, 0.9848078 + 0.1002558j]])
y20 = np.array([[0.9848078, -0.1736482], [0.1736482, 0.9848078]])
z20 = np.array([[0.9848078 - 0.1736482j, 0j], [0j, 0.9848078 + 0.1736482j]])
xz20 = np.array([[0.9848078 - 0.1227878j, -0.1227878j], [-0.1227878j, 0.9848078 + 0.1227878j]])
yz20 = np.array([[0.9848078 - 0.1227878j, -0.1227878], [0.1227878, 0.9848078 + 0.1227878j]])

lista_de_matrices = [
    {"name": "x20", "matrix": x20},
    {"name": "xy20", "matrix": xy20},
    {"name": "xyz20", "matrix": xyz20},
    {"name": "y20", "matrix": y20},
    {"name": "z20", "matrix": z20},
    {"name": "xz20", "matrix": xz20},
    {"name": "yz20", "matrix": yz20}
]

# Ejecutar el traductor
compuertas_traducidas = traducir_lista_a_quirk_gates(lista_de_matrices)

# Mostrar resultados limpios por consola
print("=== TRADUCCIÓN A COMPUERTAS DE QUIRK ===")
for nombre_compuerta, gates in compuertas_traducidas.items():
    print(f"Compuerta custom '{nombre_compuerta.ljust(6)}' -> Secuencia en Quirk: {gates}")


=== TRADUCCIÓN A COMPUERTAS DE QUIRK ===
Compuerta custom 'x20   ' -> Secuencia en Quirk: ['Z^0.5', 'Y^0.1111', 'Z^-0.5']
Compuerta custom 'xy20  ' -> Secuencia en Quirk: ['Z^0.25', 'Y^0.1111', 'Z^-0.25']
Compuerta custom 'xyz20 ' -> Secuencia en Quirk: ['Z^0.2823', 'Y^0.0906', 'Z^-0.2177']
Compuerta custom 'y20   ' -> Secuencia en Quirk: ['Y^0.1111']
Compuerta custom 'z20   ' -> Secuencia en Quirk: ['Z^0.1111']
Compuerta custom 'xz20  ' -> Secuencia en Quirk: ['Z^0.5395', 'Y^0.0784', 'Z^-0.4605']
Compuerta custom 'yz20  ' -> Secuencia en Quirk: ['Z^0.0395', 'Y^0.0784', 'Z^0.0395']


In [25]:
import json
import re
from urllib.parse import unquote

import numpy as np


def _parse_matriz_quirk(matrix_str):
    """
    Convierte la matriz de un gate custom de Quirk ("{{a,b},{c,d}}") a ndarray.

    - Quirk usa el sufijo 'i' para la unidad imaginaria: se traduce a 'j'.
    - Las filas se separan con split (no con un regex codicioso, que metía la
      llave inicial en el primer grupo y rompía complex()).
    """
    cuerpo = matrix_str.strip().strip("{}")
    filas = [f for f in re.split(r"\}\s*,\s*\{", cuerpo) if f.strip()]
    datos = [
        [complex(valor.strip().strip("{}").replace("i", "j")) for valor in fila.split(",")]
        for fila in filas
    ]
    return np.array(datos, dtype=complex)


def traducir_url_quirk_a_gates(url):
    """
    Extrae el JSON de 'circuit=' de una URL de Quirk y descompone la matriz de
    cada gate custom (U(2) de 1 qubit) en la secuencia Z^a -> Y^b -> Z^c de Quirk.

    OJO con la convención de exponentes: Quirk define Z^t = diag(1, e^{i2πt}),
    Y^t = e^{iπt}·Ry(2πt) y X^t = e^{iπt}·Rx(2πt) (src/gates/PoweringGates.js).
    Por eso Z^0.25 = S y Z^0.5 = Z, a diferencia de Qiskit, donde Z^t = e^{iπt}.
    Los exponentes se calculan en CICLOS (ángulo / 2π), no en medias vueltas.

    Devuelve {nombre_del_gate: [secuencias]}.
    """
    if "circuit=" not in url:
        raise ValueError("La URL proporcionada no contiene un parámetro 'circuit=' válido.")

    # split en vez de urlparse: urlparse rompe con el JSON de Quirk (llaves y '&').
    parte_circuit = url.split("circuit=", 1)[1].split("&", 1)[0].split("#", 1)[0]
    circuit_data = json.loads(unquote(parte_circuit))

    resultados = {}
    for gate in circuit_data.get("gates", []):
        nombre = gate.get("name") or gate.get("id")
        matrix_str = gate.get("matrix")

        # Gates definidos por 'circuit' (no por 'matrix'): no hay U que descomponer.
        if not matrix_str:
            resultados[nombre] = ["<sin matriz: gate definido por circuit>"]
            continue

        U = _parse_matriz_quirk(matrix_str)

        if U.shape != (2, 2):
            resultados[nombre] = [f"<{U.shape[0]}x{U.shape[1]}: no descomponible como ZYZ de 1 qubit>"]
            continue
        if not np.allclose(U @ U.conj().T, np.eye(2), atol=1e-6):
            resultados[nombre] = ["<matriz no unitaria>"]
            continue

        # --- Descomposición U = e^{i·fase} · Rz(phi) · Ry(theta) · Rz(lambda) ---
        U_su2 = U / np.sqrt(np.linalg.det(U))
        a, b = U_su2[0, 0], U_su2[0, 1]

        theta = 2 * np.arctan2(np.abs(b), np.abs(a))

        if np.isclose(np.abs(b), 0):
            phi = 0.0
            lambda_ = -2 * np.angle(a)
        else:
            angle_a = np.angle(a)
            angle_minus_b = np.angle(-b)
            phi = -angle_a - angle_minus_b
            lambda_ = -angle_a + angle_minus_b

        def _en_ciclos(angulo):
            """Convierte un ángulo (rad) al exponente t de Quirk: t = ángulo / 2π."""
            t = angulo / (2 * np.pi)
            return round(float((t + 0.5) % 1.0 - 0.5), 6)  # normaliza a (-0.5, 0.5]

        # El circuito se aplica de izquierda a derecha: Z(lambda) -> Y(theta) -> Z(phi).
        exponentes = [("Z", _en_ciclos(lambda_)), ("Y", _en_ciclos(theta)), ("Z", _en_ciclos(phi))]
        secuencia = [f"{etiqueta}^{t:g}" for etiqueta, t in exponentes if abs(t) > 1e-4]
        resultados[nombre] = secuencia or ["Identidad"]

    return resultados




=== COMPUERTAS TRADUCIDAS EXITOSAMENTE ===
Gate 'x20   ' -> ['Z^0.25', 'Y^0.055556', 'Z^-0.25']

Fase global: 0.174534 rad (Quirk la ignora)
Error máximo vs original: 1.34e-06  (OK)


In [28]:
# --- PRUEBA CON TU URL EXACTA ---
# url_usuario = "https://algassert.com/quirk#circuit={%22cols%22:[[%22~2onn%22]],%22gates%22:[{%22id%22:%22~2onn%22,%22name%22:%22x20%22,%22matrix%22:%22{{0.9848078,-0.1736482i},{-0.1736482i,0.9848078}}%22}]}"

url_usuario = "https://algassert.com/quirk#circuit={%22cols%22:[[%22~2onn%22]],%22gates%22:[{%22id%22:%22~2onn%22,%22name%22:%22x20%22,%22matrix%22:%22{{0.9848078,-0.1736482i},{-0.1736482i,0.9848078}}%22},{%22id%22:%22~1tvc%22,%22name%22:%22xy20%22,%22matrix%22:%22{{0.9848078,-0.1227878-0.1227878i},{0.1227878-0.1227878i,0.9848078}}%22},{%22id%22:%22~egdf%22,%22name%22:%22xyz20%22,%22matrix%22:%22{{0.9848078-0.1002558i,-0.1002558-0.1002558i},{0.1002558-0.1002558i,0.9848078+0.1002558i}}%22},{%22id%22:%22~pt7e%22,%22name%22:%22y20%22,%22matrix%22:%22{{0.9848078,-0.1736482},{0.1736482,0.9848078}}%22},{%22id%22:%22~aebp%22,%22name%22:%22z20%22,%22matrix%22:%22{{0.9848078-0.1736482i,0},{0,0.9848078+0.1736482i}}%22},{%22id%22:%22~2sp8%22,%22name%22:%22xz20%22,%22matrix%22:%22{{0.9848078-0.1227878i,-0.1227878i},{-0.1227878i,0.9848078+0.1227878i}}%22},{%22id%22:%22~15ma%22,%22name%22:%22yz20%22,%22matrix%22:%22{{0.9848078-0.1227878i,-0.1227878},{0.1227878,0.9848078+0.1227878i}}%22}]}"

traduccion = traducir_url_quirk_a_gates(url_usuario)

print("=== COMPUERTAS TRADUCIDAS EXITOSAMENTE ===")
for gate_name, secuencia in traduccion.items():
    print(f"Gate '{gate_name.ljust(6)}' -> {secuencia}")


# --- Verificación: reconstruir la matriz desde la secuencia ---
# Se replican las definiciones exactas de Quirk (mismas que usa el simulador).
def _z(t):
    return np.array([[1, 0], [0, np.exp(2j * np.pi * t)]])


def _y(t):
    phi = np.pi * t
    return np.exp(1j * phi) * np.array(
        [[np.cos(phi), -np.sin(phi)], [np.sin(phi), np.cos(phi)]]
    )


# Las compuertas de Quirk están definidas salvo fase global, por eso se compara
# hasta una fase: M / fase debe reproducir U_original.
U_original = _parse_matriz_quirk(json.loads(unquote(url_usuario.split("circuit=", 1)[1]))["gates"][0]["matrix"])

M = np.eye(2, dtype=complex)
for g in traduccion["x20"]:
    etiqueta, t = g.split("^")
    M = (_z if etiqueta == "Z" else _y)(float(t)) @ M

fase = np.trace(M @ U_original.conj().T) / 2
error = np.max(np.abs(M / fase - U_original))
print(f"\nFase global: {np.angle(fase):.6f} rad (Quirk la ignora)")
print(f"Error máximo vs original: {error:.2e}  ({'OK' if error < 1e-4 else 'REVISAR'})")


=== COMPUERTAS TRADUCIDAS EXITOSAMENTE ===
Gate 'x20   ' -> ['Z^0.25', 'Y^0.055556', 'Z^-0.25']
Gate 'xy20  ' -> ['Z^0.125', 'Y^0.055556', 'Z^-0.125']
Gate 'xyz20 ' -> ['Z^0.141147', 'Y^0.045284', 'Z^-0.108853']
Gate 'y20   ' -> ['Y^0.055556']
Gate 'z20   ' -> ['Z^0.055556']
Gate 'xz20  ' -> ['Z^0.269742', 'Y^0.039183', 'Z^-0.230258']
Gate 'yz20  ' -> ['Z^0.019742', 'Y^0.039183', 'Z^0.019742']

Fase global: 0.174534 rad (Quirk la ignora)
Error máximo vs original: 1.34e-06  (OK)


In [26]:
# --- Exportar las compuertas traducidas como URL de Quirk ---
from urllib.parse import unquote
from IPython.display import HTML, display
from qtrans.utils.url_utils import encode_quirk_url

BASE_QUIRK = "https://algassert.com/quirk#"


def traduccion_a_url_quirk(traduccion, base=BASE_QUIRK):
    """
    Convierte el dict {gate: [secuencias]} del traductor en una URL de Quirk.
    Cada gate traducido va a un cable distinto, una columna por exponente.
    Devuelve la URL con el JSON escapado (segura para un href HTML).
    """
    secuencias = [(n, [g for g in s if "^" in g]) for n, s in traduccion.items()]
    secuencias = [(n, s) for n, s in secuencias if s]
    if not secuencias:
        raise ValueError("No hay secuencias con exponentes para exportar.")

    cols = []
    for i in range(max(len(s) for _, s in secuencias)):
        col = []
        for q, (_, seq) in enumerate(secuencias):
            if i < len(seq):
                col.extend([1] * (q - len(col)))  # avanza cables hasta el cable q
                col.append(seq[i])
        cols.append(col)

    return encode_quirk_url(
        base + "circuit=" + json.dumps({"cols": cols}, separators=(",", ":"))
    )


def secuencia_inversa(secuencia):
    """
    Inverso de una secuencia de Quirk: se recorren los gates al revés y se
    niega cada exponente, porque los exponentes son aditivos (Z^a · Z^b = Z^(a+b)).
    """
    inversa = []
    for gate in reversed(secuencia):
        etiqueta, t = gate.split("^", 1)
        t = float(t)
        if abs(t) > 1e-12:
            inversa.append(f"{etiqueta}^{-t:g}")
    return inversa


def mostrar_url(titulo, url):
    """
    Imprime la URL legible (JSON crudo, sin %7B) y agrega el link con la URL
    escapada, que es la única forma segura dentro de un atributo href: las
    comillas dobles del JSON cerrarían el atributo.
    """
    legible = encode_quirk_url(url, legible=True)
    print(f"=== {titulo} ===")
    print(legible)
    display(HTML(f'<a href="{url}" target="_blank">Abrir en Quirk</a>'))
    return legible


# 1) URL con la traducción (gates nativos, sin gates custom)
url_quirk = traduccion_a_url_quirk(traduccion)
mostrar_url("URL DE LA TRADUCCIÓN", url_quirk)


# 2) URL de verificación: gate original + inverso de la traducción -> debe dar 100% |0>
circuito_original = json.loads(unquote(url_usuario.split("circuit=", 1)[1]))
cols_verificacion = list(circuito_original["cols"])
for q, (nombre, secuencia) in enumerate(traduccion.items()):
    for gate in secuencia_inversa([g for g in secuencia if "^" in g]):
        cols_verificacion.append([1] * q + [gate])

url_verificacion = encode_quirk_url(
    BASE_QUIRK
    + "circuit="
    + json.dumps(
        {"cols": cols_verificacion, "gates": circuito_original.get("gates", [])},
        separators=(",", ":"),
    )
)
mostrar_url("URL DE VERIFICACIÓN (debe quedar 100% |0>)", url_verificacion)


=== URL DE LA TRADUCCIÓN ===
https://algassert.com/quirk#circuit={"cols":[["Z^0.25"],["Y^0.055556"],["Z^-0.25"]]}


=== URL DE VERIFICACIÓN (debe quedar 100% |0>) ===
https://algassert.com/quirk#circuit={"cols":[["~2onn"],["Z^0.25"],["Y^-0.055556"],["Z^-0.25"]],"gates":[{"id":"~2onn","name":"x20","matrix":"{{0.9848078,-0.1736482i},{-0.1736482i,0.9848078}}"}]}


'https://algassert.com/quirk#circuit={"cols":[["~2onn"],["Z^0.25"],["Y^-0.055556"],["Z^-0.25"]],"gates":[{"id":"~2onn","name":"x20","matrix":"{{0.9848078,-0.1736482i},{-0.1736482i,0.9848078}}"}]}'

In [27]:

# 1) URL con la traducción (gates nativos, sin gates custom)
url_quirk = traduccion_a_url_quirk(traduccion)
print("=== URL DE LA TRADUCCIÓN ===")
print(url_quirk)
display(HTML(f'<a href="{url_quirk}" target="_blank">Abrir traducción en Quirk</a>'))



=== URL DE LA TRADUCCIÓN ===
https://algassert.com/quirk#circuit=%7B%22cols%22%3A%5B%5B%22Z%5E0.25%22%5D%2C%5B%22Y%5E0.055556%22%5D%2C%5B%22Z%5E-0.25%22%5D%5D%7D


In [ ]:

# 2) URL de verificación: gate original + inverso de la traducción -> debe dar 100% |0>
circuito_original = json.loads(unquote(url_usuario.split("circuit=", 1)[1]))
cols_verificacion = list(circuito_original["cols"])
for q, (nombre, secuencia) in enumerate(traduccion.items()):
    for gate in secuencia_inversa([g for g in secuencia if "^" in g]):
        cols_verificacion.append([1] * q + [gate])

url_verificacion = encode_quirk_url(
    BASE_QUIRK
    + "circuit="
    + json.dumps(
        {"cols": cols_verificacion, "gates": circuito_original.get("gates", [])},
        separators=(",", ":"),
    )
)
print("\n=== URL DE VERIFICACIÓN (debe quedar 100% |0>) ===")
print(url_verificacion)
display(HTML(f'<a href="{url_verificacion}" target="_blank">Abrir verificación en Quirk</a>'))
